# AfiLearn University Learning Analytics Office -  Big-Data Processing

Processes `studentVle.csv` (10,655,280 rows, largest table in OULAD) with
PySpark to produce useful engagement aggregates.



In [1]:
#create a spark session 
import os
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = (
    SparkSession.builder
    .appName("AfiLearn-studentVle-aggregation")
    .master("local[*]")
    .config("spark.driver.memory", "4g")          
    .config("spark.sql.shuffle.partitions", "32") 
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

log4j = spark._jvm.org.apache.log4j
log4j.LogManager.getLogger("org.apache.spark.sql.catalyst.expressions.RowBasedKeyValueBatch").setLevel(log4j.Level.ERROR)
spark.sparkContext.setLogLevel("WARN")
spark

:: loading settings :: url = jar:file:/Users/gyauk/codetools/Spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/gyauk/.ivy2/cache
The jars for the packages stored in: /Users/gyauk/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3b8d7a96-9e42-4e82-ba88-ae25bc38bc8b;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.5 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.5 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 1099ms :: artifacts dl 28ms
	:

## Load the raw file

`studentVle.csv` has six columns: `code_module`, `code_presentation`, `id_student`,
`id_site`, `date` (a day-offset from the presentation start, can be negative), and
`sum_click`. An explicit schema is used rather than inference would require a
full extra pass over 10.6M rows just to guess types.

In [4]:
BASE = "data"  # adjust to wherever the OULAD CSVs live

schema = StructType([
    StructField("code_module", StringType(), False),
    StructField("code_presentation", StringType(), False),
    StructField("id_student", IntegerType(), False),
    StructField("id_site", IntegerType(), False),
    StructField("date", IntegerType(), False),
    StructField("sum_click", IntegerType(), False),
])

raw = spark.read.csv(f"{BASE}/studentVle.csv", header=True, schema=schema)
raw_count = raw.count()
print(f"raw rows = {raw_count:,}")

raw rows = 10,655,280


## Resolve the duplicate key

Analyzing the files we found that the natural key
(`code_module`, `code_presentation`, `id_student`, `id_site`, `date`) is **not** unique in
the raw data about 21% of rows share a key with another row, that is the same student
clicking the same resource on the same day is split across multiple rows instead of one
row with `sum_click` already totalled. This step collapses that down to the true grain by
summing `sum_click` over the key. 

In [5]:
key_cols = ["code_module", "code_presentation", "id_student", "id_site", "date"]

clean = raw.groupBy(*key_cols).agg(F.sum("sum_click").alias("sum_click")).cache()
clean_count = clean.count()
print(f"clean (deduplicated) rows = {clean_count:,}")
print(f"duplicate rows collapsed = {raw_count - clean_count:,} ({(raw_count - clean_count) / raw_count:.1%})")

clean (deduplicated) rows = 8,459,320
duplicate rows collapsed = 2,195,960 (20.6%)


## Three aggregates

- **student engagement summary** 
- **Resource popularity** 
- **Weekly engagement trend** 

In [6]:
#Student engagement summary
student_summary = (
    clean.groupBy("code_module", "code_presentation", "id_student")
    .agg(
        F.sum("sum_click").alias("total_clicks"),
        F.countDistinct("date").alias("active_days"),
        F.countDistinct("id_site").alias("distinct_resources"),
        F.min("date").alias("first_active_day"),
        F.max("date").alias("last_active_day"),
    )
)
print(f"student_summary rows = {student_summary.count():,}")
student_summary.orderBy(F.desc("total_clicks")).show(5, truncate=False)

student_summary rows = 29,228


+-----------+-----------------+----------+------------+-----------+------------------+----------------+---------------+
|code_module|code_presentation|id_student|total_clicks|active_days|distinct_resources|first_active_day|last_active_day|
+-----------+-----------------+----------+------------+-----------+------------------+----------------+---------------+
|CCC        |2014J            |80868     |24139       |278        |142               |-18             |269            |
|FFF        |2013B            |517269    |21123       |212        |334               |-3              |233            |
|CCC        |2014J            |611417    |20391       |205        |114               |-18             |269            |
|FFF        |2013J            |368315    |19734       |216        |316               |-18             |266            |
|FFF        |2014J            |583487    |19461       |270        |199               |-18             |269            |
+-----------+-----------------+---------

In [7]:
#Resource popularity
resource_popularity = (
    clean.groupBy("code_module", "code_presentation", "id_site")
    .agg(
        F.sum("sum_click").alias("total_clicks"),
        F.countDistinct("id_student").alias("distinct_students"),
    )
)
print(f"resource_popularity rows = {resource_popularity.count():,}")
resource_popularity.orderBy(F.desc("total_clicks")).show(5, truncate=False)

resource_popularity rows = 6,268


+-----------+-----------------+-------+------------+-----------------+
|code_module|code_presentation|id_site|total_clicks|distinct_students|
+-----------+-----------------+-------+------------+-----------------+
|FFF        |2013J            |716238 |718737      |2096             |
|FFF        |2014J            |882537 |716869      |2115             |
|FFF        |2013B            |526721 |586632      |1510             |
|DDD        |2013J            |673519 |455896      |1767             |
|CCC        |2014J            |909013 |442887      |2300             |
+-----------+-----------------+-------+------------+-----------------+
only showing top 5 rows



In [8]:
#Weekly engagement trend
weekly_trend = (
    clean.withColumn("presentation_week", F.floor(F.col("date") / F.lit(7)))
    .groupBy("code_module", "code_presentation", "presentation_week")
    .agg(
        F.sum("sum_click").alias("total_clicks"),
        F.countDistinct("id_student").alias("distinct_students"),
    )
)
print(f"weekly_trend rows = {weekly_trend.count():,}")

# Example: module AAA, presentation 2013J
weekly_trend.filter(
    (F.col("code_module") == "AAA") & (F.col("code_presentation") == "2013J")
).orderBy("presentation_week").show(15, truncate=False)

weekly_trend rows = 879


+-----------+-----------------+-----------------+------------+-----------------+
|code_module|code_presentation|presentation_week|total_clicks|distinct_students|
+-----------+-----------------+-----------------+------------+-----------------+
|AAA        |2013J            |-2               |21100       |301              |
|AAA        |2013J            |-1               |28523       |313              |
|AAA        |2013J            |0                |33960       |337              |
|AAA        |2013J            |1                |38066       |341              |
|AAA        |2013J            |2                |44171       |357              |
|AAA        |2013J            |3                |23292       |305              |
|AAA        |2013J            |4                |19847       |315              |
|AAA        |2013J            |5                |20532       |312              |
|AAA        |2013J            |6                |19382       |318              |
|AAA        |2013J          

**What this shows for AAA/2013J**  
- Engagement peaks around presentation-week 2 at roughly 44,000 weekly clicks
across 357 active students, then declines steadily to under 7,000 clicks by week 11. Indicating  a
drop-off curve. A presentation whose engagement falls off unusually early, relative to this baseline
shape, is worth flagging to the office.

## Outputs

Split across two folders matching the access tiers above, so the restriction is enforced by
where the file lives, not just by policy on paper. `restricted/` should get tighter
filesystem/db permissions than `aggregates/` when this lands in the shared environment.

In [ ]:
OUT = "./output"

#Output to parquet
student_summary.write.mode("overwrite").parquet(f"{OUT}/aggregates/student_engagement_summary")
resource_popularity.write.mode("overwrite").parquet(f"{OUT}/aggregates/resource_popularity")
weekly_trend.write.mode("overwrite").parquet(f"{OUT}/aggregates/weekly_engagement_trend")


#output to csv 
student_summary.write.mode("overwrite").csv(f"{OUT}/aggregates/student_engagement_summary")
resource_popularity.write.mode("overwrite").csv(f"{OUT}/aggregates/resource_popularity")
weekly_trend.write.mode("overwrite").csv(f"{OUT}/aggregates/weekly_engagement_trend")


print("All three aggregates written.")
print(f" aggregates/student_engagement_summary : {student_summary.count():,} rows")
print(f" aggregates/resource_popularity         : {resource_popularity.count():,} rows")
print(f" aggregates/weekly_engagement_trend     : {weekly_trend.count():,} rows")

All three aggregates written.


 restricted/student_engagement_summary : 29,228 rows


 aggregates/resource_popularity         : 6,268 rows


 aggregates/weekly_engagement_trend     : 879 rows


In [10]:
spark.stop()